# Clustering Spectral — Communes françaises

Ce notebook applique une approche de **clustering spectral** sur le même jeu de données que les analyses DBSCAN et CAH précédentes.

### Plan
1. Préparation des données (identique aux notebooks précédents)
2. Pourquoi le clustering spectral ?
3. Choix du K via l'eigengap (valeurs propres du Laplacien)
4. Sensibilité au paramètre `n_neighbors`
5. Comparaison multi-K avec métriques
6. Application sur données complètes
7. Profil et analyse des clusters
8. Visualisation cartographique
9. Comparaison avec CAH et DBSCAN

---
## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import SpectralClustering
from sklearn.neighbors import kneighbors_graph, KNeighborsClassifier
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA

import scipy.sparse as sp
import scipy.sparse.linalg as spla
import scipy.cluster.hierarchy as sch

import geopandas as gpd
import warnings
warnings.filterwarnings('ignore')

print('Imports OK')

---
## 1. Préparation des données

Identique aux notebooks DBSCAN et CAH — on part du même `data` brut.

In [ ]:
df_travail = data.copy()

colonnes_id = ['codecommune', 'dep']
for col in colonnes_id:
    if col in df_travail.columns:
        df_travail = df_travail.set_index(col, append=True)

df_numerique = (
    df_travail.select_dtypes(include=[np.number])
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

scaler = StandardScaler()
matrice_scaled = scaler.fit_transform(df_numerique)
donnees_clustering = pd.DataFrame(
    matrice_scaled,
    columns=df_numerique.columns,
    index=df_numerique.index
)

print(f'Données prêtes : {donnees_clustering.shape[0]:,} communes x {donnees_clustering.shape[1]} variables')

In [ ]:
# Sous-échantillon pour les étapes exploratoires coûteuses
# SpectralClustering est O(n²) en mémoire → sample pour le réglage des hyperparamètres

SAMPLE_STEP = 10
donnees_sample = donnees_clustering[::SAMPLE_STEP].copy()

print(f'Sample exploratoire : {donnees_sample.shape[0]:,} communes (1/{SAMPLE_STEP})')
print(f'Données complètes   : {donnees_clustering.shape[0]:,} communes')

---
## 2. Pourquoi le clustering spectral ?

Le clustering spectral repose sur la **théorie des graphes** plutôt que sur des distances euclidiennes.

| Méthode | Forme des clusters | Bruit | Complexité |
|---|---|---|---|
| K-Means | Sphérique (convexe) | ✗ | O(n·k·iter) |
| CAH Ward | Ellipsoïdale | ✗ | O(n² log n) |
| DBSCAN | Quelconque | ✓ | O(n log n) |
| **Spectral** | **Quelconque (non-convexe)** | Partiel | O(n²) à O(n³) |

### Principe en 3 étapes
1. **Graphe de similarité** : chaque commune est reliée à ses `n_neighbors` plus proches voisins
2. **Laplacien normalisé** : `L = I − D⁻¹/² A D⁻¹/²` encode la structure de connectivité du graphe
3. **K-Means spectral** : les K premiers vecteurs propres de L forment un embedding dans lequel les clusters deviennent séparables linéairement

> **Intuition** : là où K-Means cherche des boules, le clustering spectral cherche des *communautés connectées* dans un graphe — comme détecter des îles sur une carte.

---
## 3. Choix du K optimal — méthode eigengap

Le nombre de clusters K correspond au **plus grand saut** entre deux valeurs propres consécutives du Laplacien normalisé.
Si K clusters bien séparés existent, les K premières valeurs propres sont proches de 0, puis un saut net apparaît.

In [ ]:
def compute_laplacian_eigenvalues(X, n_neighbors=10, n_eigvals=15):
    """
    Construit le graphe k-NN symétrique, calcule le Laplacien normalisé
    et retourne ses premières valeurs propres triées.
    """
    A = kneighbors_graph(X, n_neighbors=n_neighbors,
                         mode='connectivity', include_self=False)
    A = A + A.T
    A.data = np.ones_like(A.data)  # poids binaires

    degrees = np.array(A.sum(axis=1)).flatten()
    D_inv_sqrt = sp.diags(1.0 / np.sqrt(np.maximum(degrees, 1e-10)))

    # Laplacien normalisé : L_sym = I - D^{-1/2} A D^{-1/2}
    L_sym = sp.eye(A.shape[0]) - D_inv_sqrt @ A @ D_inv_sqrt

    eigenvalues, _ = spla.eigsh(L_sym, k=n_eigvals, which='SM')
    return np.sort(np.real(eigenvalues))


N_NEIGHBORS = 10
N_EIGVALS   = 15

print(f'Calcul des valeurs propres (n_neighbors={N_NEIGHBORS})...')
eigenvalues = compute_laplacian_eigenvalues(
    donnees_sample.values, n_neighbors=N_NEIGHBORS, n_eigvals=N_EIGVALS
)
print('Terminé.')
print('Valeurs propres :', np.round(eigenvalues, 4))

In [ ]:
gaps = np.diff(eigenvalues)
K_eigengap = int(np.argmax(gaps)) + 1  # indice avant le plus grand saut → K suggéré

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(range(1, len(eigenvalues)+1), eigenvalues, 'o-',
        color='steelblue', linewidth=2, markersize=6)
ax.axvline(x=K_eigengap, color='red', linestyle='--', linewidth=1.5,
           label=f'K suggéré = {K_eigengap}')
ax.set_xlabel('Indice de la valeur propre')
ax.set_ylabel('Valeur propre λ')
ax.set_title('Spectre du Laplacien normalisé')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
colors_bar = ['red' if i == K_eigengap - 1 else 'steelblue' for i in range(len(gaps))]
ax.bar(range(1, len(gaps)+1), gaps, color=colors_bar, alpha=0.8)
ax.set_xlabel('k')
ax.set_ylabel('λ(k+1) − λ(k)')
ax.set_title('Eigengap — saut entre valeurs propres consécutives')
ax.legend(handles=[mpatches.Patch(color='red', label=f'Plus grand saut → K={K_eigengap}')])
ax.grid(alpha=0.3)

plt.suptitle('Choix de K par la méthode eigengap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\n→ K suggéré par l\'eigengap : {K_eigengap}')

---
## 4. Sensibilité au paramètre `n_neighbors`

`n_neighbors` contrôle la densité du graphe de similarité.
- Trop petit → graphe fragmenté, sur-segmentation
- Trop grand → les structures locales sont noyées

On balaye plusieurs valeurs pour choisir la plus robuste.

In [ ]:
K_TEST           = 5   # valeur de K pour ce test de sensibilité
neighbors_range  = [5, 10, 15, 20, 30]

resultats_neighbors = []

for nn in neighbors_range:
    model = SpectralClustering(
        n_clusters=K_TEST,
        affinity='nearest_neighbors',
        n_neighbors=nn,
        assign_labels='kmeans',
        random_state=42,
        n_jobs=-1
    )
    labels = model.fit_predict(donnees_sample.values)

    sil = silhouette_score(donnees_sample.values, labels, sample_size=2000, random_state=42)
    db  = davies_bouldin_score(donnees_sample.values, labels)
    ch  = calinski_harabasz_score(donnees_sample.values, labels)

    resultats_neighbors.append({
        'n_neighbors': nn,
        'silhouette': round(sil, 4),
        'davies_bouldin': round(db, 4),
        'calinski_harabasz': round(ch, 1)
    })
    print(f'n_neighbors={nn:2d} | Silhouette={sil:.4f} | DB={db:.4f} | CH={ch:.1f}')

df_neighbors = pd.DataFrame(resultats_neighbors).set_index('n_neighbors')
df_neighbors

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics = [
    ('silhouette',        'Silhouette (↑)',        'green'),
    ('davies_bouldin',    'Davies-Bouldin (↓)',     'red'),
    ('calinski_harabasz', 'Calinski-Harabasz (↑)',  'steelblue'),
]

for ax, (col, title, color) in zip(axes, metrics):
    ax.plot(df_neighbors.index, df_neighbors[col], 'o-', color=color, linewidth=2)
    ax.set_xlabel('n_neighbors')
    ax.set_title(title)
    ax.grid(alpha=0.3)

plt.suptitle(f'Sensibilité à n_neighbors (K={K_TEST})', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Comparaison multi-K avec métriques

On fixe le `n_neighbors` retenu à l'étape précédente et on teste plusieurs valeurs de K.

In [ ]:
N_NEIGHBORS_FINAL = 10   # ← à ajuster selon l'étape 4
K_RANGE           = range(2, 9)

resultats_k = []

for k in K_RANGE:
    model = SpectralClustering(
        n_clusters=k,
        affinity='nearest_neighbors',
        n_neighbors=N_NEIGHBORS_FINAL,
        assign_labels='kmeans',
        random_state=42,
        n_jobs=-1
    )
    labels = model.fit_predict(donnees_sample.values)

    sil = silhouette_score(donnees_sample.values, labels, sample_size=2000, random_state=42)
    db  = davies_bouldin_score(donnees_sample.values, labels)
    ch  = calinski_harabasz_score(donnees_sample.values, labels)

    resultats_k.append({'K': k, 'silhouette': sil, 'davies_bouldin': db,
                        'calinski_harabasz': ch})
    print(f'K={k} | Silhouette={sil:.4f} | DB={db:.4f} | CH={ch:.1f}')

df_k = pd.DataFrame(resultats_k).set_index('K')
df_k.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics = [
    ('silhouette',        'Silhouette (↑)',        'green'),
    ('davies_bouldin',    'Davies-Bouldin (↓)',     'red'),
    ('calinski_harabasz', 'Calinski-Harabasz (↑)',  'steelblue'),
]

for ax, (col, title, color) in zip(axes, metrics):
    ax.plot(df_k.index, df_k[col], 'o-', color=color, linewidth=2)
    ax.axvline(x=K_eigengap, color='orange', linestyle='--', linewidth=1.5,
               label=f'Eigengap → K={K_eigengap}')
    ax.set_xlabel('K')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Métriques de clustering selon K', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Choix final ────────────────────────────────────────────────────
K_FINAL = 5   # ← modifiez selon les résultats ci-dessus
print(f'K retenu pour la suite : {K_FINAL}')

---
## 6. Application du clustering spectral — données complètes

> **Note sur la scalabilité** : `SpectralClustering` de sklearn construit une matrice d'affinité dense (O(n²) en mémoire).  
> Pour de grands datasets, on utilise `affinity='nearest_neighbors'` (sparse) avec `assign_labels='cluster_qr'`.  
> Si n > 10 000, on applique le clustering sur le sample puis on propage les labels par k-NN.

In [ ]:
N = donnees_clustering.shape[0]
print(f'Nombre total de communes : {N:,}')

if N <= 10_000:
    print('Dataset de taille raisonnable → clustering direct sur toutes les données.')
    model_final = SpectralClustering(
        n_clusters=K_FINAL,
        affinity='nearest_neighbors',
        n_neighbors=N_NEIGHBORS_FINAL,
        assign_labels='cluster_qr',   # plus stable numériquement que kmeans
        random_state=42,
        n_jobs=-1
    )
    labels_spectral = model_final.fit_predict(donnees_clustering.values)

else:
    print(f'Grand dataset ({N:,} communes) → clustering sur sample + propagation k-NN.')

    # 1. Spectral sur le sample
    model_sample = SpectralClustering(
        n_clusters=K_FINAL,
        affinity='nearest_neighbors',
        n_neighbors=N_NEIGHBORS_FINAL,
        assign_labels='cluster_qr',
        random_state=42,
        n_jobs=-1
    )
    labels_sample = model_sample.fit_predict(donnees_sample.values)

    # 2. Propagation : chaque commune hors-sample hérite du label
    #    de son plus proche voisin dans le sample
    knn_prop = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
    knn_prop.fit(donnees_sample.values, labels_sample)
    labels_spectral = knn_prop.predict(donnees_clustering.values)
    labels_spectral[::SAMPLE_STEP] = labels_sample   # cohérence garantie sur le sample

# Rattachement au dataframe
donnees_clustering['label_spectral'] = labels_spectral

print('\nDistribution des clusters :')
dist = pd.Series(labels_spectral).value_counts().sort_index()
for k, n in dist.items():
    print(f'  Cluster {k} : {n:,} communes ({n/N*100:.1f}%)')

In [ ]:
# ── Métriques finales (évaluées sur un sample pour la rapidité) ──
idx_eval = np.random.choice(N, size=min(5000, N), replace=False)
X_eval   = donnees_clustering.drop(columns='label_spectral').values[idx_eval]
y_eval   = labels_spectral[idx_eval]

sil_final = silhouette_score(X_eval, y_eval)
db_final  = davies_bouldin_score(X_eval, y_eval)
ch_final  = calinski_harabasz_score(X_eval, y_eval)

print(f'Métriques finales  (K={K_FINAL}, n_neighbors={N_NEIGHBORS_FINAL})')
print(f'  Silhouette        : {sil_final:.4f}  (↑ mieux, max=1)')
print(f'  Davies-Bouldin    : {db_final:.4f}  (↓ mieux, min=0)')
print(f'  Calinski-Harabasz : {ch_final:.1f} (↑ mieux)')

---
## 7. Analyse des clusters — profils et visualisation

In [ ]:
# ── Profils moyens par cluster (variables originales) ─────────────
df_profil = df_numerique.copy()
df_profil['label_spectral'] = labels_spectral

profils = df_profil.groupby('label_spectral').mean()
profils.round(3)

In [ ]:
# ── Heatmap des profils standardisés ─────────────────────────────
profils_std = profils.apply(lambda col: (col - col.mean()) / col.std(), axis=0)

fig_w = max(12, len(profils.columns) * 0.8)
plt.figure(figsize=(fig_w, K_FINAL + 2))
sns.heatmap(
    profils_std,
    cmap='RdBu_r', center=0, annot=True, fmt='.2f',
    linewidths=0.5,
    cbar_kws={'label': 'Écart à la moyenne (σ)'}
)
plt.title(f'Profil moyen par cluster spectral (K={K_FINAL}) — valeurs standardisées',
          fontsize=12, fontweight='bold')
plt.ylabel('Cluster')
plt.tight_layout()
plt.show()

In [ ]:
# ── Projection PCA 2D — séparation des clusters ──────────────────
pca = PCA(n_components=2, random_state=42)
X_pca      = pca.fit_transform(donnees_sample.drop(columns='label_spectral', errors='ignore').values)
labels_pca = labels_spectral[::SAMPLE_STEP]

cmap_pca = plt.get_cmap('Set1', K_FINAL)

plt.figure(figsize=(9, 7))
for k in range(K_FINAL):
    mask = labels_pca == k
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=[mcolors.to_hex(cmap_pca(k))],
                label=f'Cluster {k}', alpha=0.5, s=15, linewidths=0)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title(f'Projection PCA — Clustering Spectral K={K_FINAL} (sample)')
plt.legend(markerscale=2, frameon=False)
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# ── Boxplots des variables les plus discriminantes ────────────────
# On calcule la variance inter-cluster pour chaque variable
variance_inter = profils_std.var(axis=0).sort_values(ascending=False)
top_vars = variance_inter.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

df_box = df_numerique[top_vars].copy()
df_box['Cluster'] = labels_spectral.astype(str)

for i, var in enumerate(top_vars):
    ax = axes[i]
    for k in range(K_FINAL):
        vals = df_box.loc[df_box['Cluster'] == str(k), var]
        ax.boxplot(vals, positions=[k], widths=0.6,
                   patch_artist=True,
                   boxprops=dict(facecolor=mcolors.to_hex(cmap_pca(k)), alpha=0.7),
                   medianprops=dict(color='black', linewidth=2),
                   flierprops=dict(marker='.', markersize=2, alpha=0.3))
    ax.set_title(var, fontsize=10)
    ax.set_xticks(range(K_FINAL))
    ax.set_xticklabels([f'C{k}' for k in range(K_FINAL)])
    ax.grid(alpha=0.3, axis='y')

plt.suptitle(f'Top 6 variables discriminantes — Clustering Spectral K={K_FINAL}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Visualisation cartographique

In [ ]:
# ── Préparation ───────────────────────────────────────────────────
df_carte = df_travail.reset_index().copy()
df_carte['codecommune'] = df_carte['codecommune'].astype(str).str.zfill(5)
df_carte['label_spectral'] = labels_spectral
df_carte['Nom_Cluster'] = df_carte['label_spectral'].apply(lambda l: f'Cluster {l}')
n_total = len(df_carte)

# ── GeoJSON ────────────────────────────────────────────────────────
url_geojson = 'https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/communes.geojson'
france_communes = gpd.read_file(url_geojson)
carte_data = france_communes.merge(df_carte, left_on='code', right_on='codecommune')

# ── Couleurs ────────────────────────────────────────────────────────
cmap_clusters    = plt.get_cmap('Set1', K_FINAL)
couleurs_dict    = {f'Cluster {i}': mcolors.to_hex(cmap_clusters(i)) for i in range(K_FINAL)}
categories_finales = [f'Cluster {i}' for i in range(K_FINAL)]
cmap_custom      = mcolors.ListedColormap([couleurs_dict[c] for c in categories_finales])

# ── Carte ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(15, 15), dpi=150)

carte_data.plot(
    column='Nom_Cluster', ax=ax,
    categorical=True, categories=categories_finales,
    cmap=cmap_custom, legend=False,
    linewidth=0, edgecolor='none',
    missing_kwds={'color': '#eeeeee', 'label': 'Données manquantes'}
)

# Légende manuelle avec effectifs
handles = []
for cat in categories_finales:
    n = (df_carte['Nom_Cluster'] == cat).sum()
    pct = n / n_total * 100
    patch = mpatches.Patch(color=couleurs_dict[cat],
                           label=f'{cat}  ({n:,} communes, {pct:.1f}%)')
    handles.append(patch)

ax.legend(
    handles=handles,
    title=f'Clustering Spectral — K={K_FINAL}, n_neighbors={N_NEIGHBORS_FINAL}',
    loc='upper left', bbox_to_anchor=(1, 1),
    frameon=False, fontsize=11, title_fontsize=12
)

ax.set_axis_off()
plt.title(f'Carte de France — Clustering Spectral (K={K_FINAL} clusters)',
          fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

---
## 9. Comparaison avec CAH et DBSCAN

On compare les métriques des trois approches sur le même échantillon.
Assurez-vous que `labels_cah` et `labels_dbscan` sont disponibles dans le notebook.

In [ ]:
# ── Assurez-vous que ces variables existent dans votre environnement ──
# labels_cah     : labels de la CAH (K=5)
# labels_dbscan  : labels DBSCAN (hors bruit, -1 exclu)
# ─────────────────────────────────────────────────────────────────────

def metriques_clustering(X, labels, nom, sample_size=3000):
    """Calcule silhouette, Davies-Bouldin et Calinski-Harabasz sur un sample."""
    mask = labels != -1   # exclut le bruit DBSCAN
    X_m  = X[mask]
    y_m  = labels[mask]
    if len(np.unique(y_m)) < 2:
        return {'Méthode': nom, 'Silhouette': np.nan,
                'Davies-Bouldin': np.nan, 'Calinski-Harabasz': np.nan}
    idx = np.random.choice(len(y_m), size=min(sample_size, len(y_m)), replace=False)
    return {
        'Méthode': nom,
        'Silhouette':         round(silhouette_score(X_m[idx], y_m[idx]), 4),
        'Davies-Bouldin':     round(davies_bouldin_score(X_m[idx], y_m[idx]), 4),
        'Calinski-Harabasz':  round(calinski_harabasz_score(X_m[idx], y_m[idx]), 1),
    }

X_full = donnees_clustering.drop(columns='label_spectral', errors='ignore').values

comparaison = []
comparaison.append(metriques_clustering(X_full, labels_spectral, f'Spectral (K={K_FINAL})'))

# Décommentez selon les méthodes disponibles :
# comparaison.append(metriques_clustering(X_full, labels_cah,    f'CAH Ward (K=5)'))
# comparaison.append(metriques_clustering(X_full, labels_dbscan, 'DBSCAN'))

df_comparaison = pd.DataFrame(comparaison).set_index('Méthode')
df_comparaison

In [ ]:
# ── Graphique de comparaison ──────────────────────────────────────
if len(df_comparaison) > 1:
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    colors_comp = plt.get_cmap('tab10', len(df_comparaison))

    for i, (col, title) in enumerate([
        ('Silhouette',        'Silhouette (↑ mieux)'),
        ('Davies-Bouldin',    'Davies-Bouldin (↓ mieux)'),
        ('Calinski-Harabasz', 'Calinski-Harabasz (↑ mieux)'),
    ]):
        ax = axes[i]
        vals   = df_comparaison[col]
        labels_comp = vals.index.tolist()
        bars = ax.bar(labels_comp, vals,
                      color=[mcolors.to_hex(colors_comp(j)) for j in range(len(vals))],
                      alpha=0.8)
        ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
        ax.set_title(title)
        ax.set_xticklabels(labels_comp, rotation=15, ha='right', fontsize=9)
        ax.grid(alpha=0.3, axis='y')

    plt.suptitle('Comparaison des méthodes de clustering', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Ajoutez les labels CAH/DBSCAN pour afficher la comparaison.')